## **ПРОИЗВОДСТВЕННАЯ ПРАКТИКА. НЕДЕЛЯ 1: ОСНОВЫ ИСПЫТАНИЙ И СЕРТИФИКАЦИИ СРЕДСТВ ЗАЩИТЫ ИНФОРМАЦИИ**

## **Задание 3. Формирование программы испытаний**

# **Проект: Автоматизированное испытание безопасности приложений (SAST)**

In [ ]:
import os
import re

# I. Тестовый файл с уязвимостями
vulnerable_code = """
import hashlib

def login_user(username, password):
    # УЯЗВИМОСТЬ 1: Жестко прописанный пароль администратора (Hardcoded credentials)
    admin_password = "SuperSecretAdminPassword2026!"

    # УЯЗВИМОСТЬ 2: Использование небезопасного алгоритма хэширования MD5 (СТ РК 1073-2025 требует минимум SHA-256)
    hashed_password = hashlib.md5(password.encode()).hexdigest()

    if username == "admin" and password == admin_password:
        return "Добро пожаловать, админ!"

    return f"Пользователь {username} авторизован с хэшем {hashed_password}"

def SaveUserData(user_id, phone, inn):
    # УЯЗВИМОСТЬ 3: Логирование конфиденциальных персональных данных в открытом виде (Нарушение Закона РК о ПД)
    print(f"[LOG] Сохранение данных пользователя {user_id}: ИИН={inn}, Телефон={phone}")
"""

with open ("auth_module.py", "w", encoding = "utf-8") as f:
  f.write(vulnerable_code)



### Результат создания `auth_module.py`

Файл `auth_module.py` успешно создан и записан на диск. Теперь он готов для анализа SAST-сканером.

### I. Создание первого тестового модуля с известными уязвимостями

Этот блок кода создает файл `auth_module.py`, который содержит несколько распространенных уязвимостей, таких как жестко прописанный пароль, использование устаревшего алгоритма хеширования (MD5) и утечка персональных данных в логи. Этот файл будет использоваться для демонстрации работы SAST-сканера.

In [ ]:
import os
import re

# II. Скрипт автоматизированного испытания безопасности (SAST-сканер)
def run_security_test(file_path):
  print ("=" * 70)
  print (f"ЗАПУСК АВТОМАТИЗИРОВАННОГО ИСПЫТАНИЯ ДЛЯ ФАЙЛА: {file_path}")
  print ("=" * 70)

  if not os.path.exists(file_path):
    print ("Файл не найден.")
    return

  # С этого места код должен быть правильно выровнен
  with open(file_path, "r", encoding = "utf-8") as f:
    lines = f.readlines()

    issues_found = 0

    # Правила проверок (На наличие уязвимостей)
    rules = {
        "Hardcoded Password": re.compile(r"(password|passwd|secret)\s*=\s*['\"][^'\"]+['\"]", re.IGNORECASE),
        "Insecure Crypto (MD5/SHA1)": re.compile(r"hashlib\.(md5|sha1)"),
        "Personal Data Leak in Logs": re.compile(r"print\(.*(иин|inn|phone|паспорт|f\s*['\"].*\{.*\}).*")
    }

    for line_num, line in enumerate(lines, 1):
      line_clean = line.strip()
      # Пропуск комментариев
      if line_clean.startswith("#"):
        continue

      for rule_name, pattern in rules.items():
        if pattern.search(line_clean):
          print(f"[КРИТИЧЕСКАЯ УЯЗВИМОСТЬ ТЕСТА] -> {rule_name}")
          print(f"Строка {line_num}: {line_clean}")
          print("-" * 70)
          issues_found += 1

    print(f"ИТОГ ИСПЫТАНИЙ: Обнаружено уязвимостей: {issues_found}")
    if issues_found > 0:
        print("РЕЗУЛЬТАТ: Проверка НЕ ПРОЙДЕНА. Требуется устранение замечаний согласно СТ РК 1073.")
    else:
        print("РЕЗУЛЬТАТ: Проверка успешно пройдена. Уязвимостей не обнаружено.")
    print("=" * 70)

# Запуск сканирования созданного модуля
run_security_test("auth_module.py")

ЗАПУСК АВТОМАТИЗИРОВАННОГО ИСПЫТАНИЯ ДЛЯ ФАЙЛА: auth_module.py
[КРИТИЧЕСКАЯ УЯЗВИМОСТЬ ТЕСТА] -> Hardcoded Password
Строка 6: admin_password = "SuperSecretAdminPassword2026!"
----------------------------------------------------------------------
[КРИТИЧЕСКАЯ УЯЗВИМОСТЬ ТЕСТА] -> Insecure Crypto (MD5/SHA1)
Строка 9: hashed_password = hashlib.md5(password.encode()).hexdigest()
----------------------------------------------------------------------
[КРИТИЧЕСКАЯ УЯЗВИМОСТЬ ТЕСТА] -> Personal Data Leak in Logs
Строка 18: print(f"[LOG] Сохранение данных пользователя {user_id}: ИИН={inn}, Телефон={phone}")
----------------------------------------------------------------------
ИТОГ ИСПЫТАНИЙ: Обнаружено уязвимостей: 3
РЕЗУЛЬТАТ: Проверка НЕ ПРОЙДЕНА. Требуется устранение замечаний согласно СТ РК 1073.


### Результат SAST-сканирования `auth_module.py`

Сканер успешно обнаружил 3 уязвимости в `auth_module.py`, соответствующие определенным правилам. Это подтверждает, что модуль содержит критические дефекты безопасности и требует доработки согласно требованиям СТ РК 1073.

### II. Запуск SAST-сканера на первом тестовом модуле

Этот блок кода содержит функцию `run_security_test`, которая имитирует работу простого SAST-сканера. Она анализирует переданный файл на наличие заданных уязвимостей (жестко прописанный пароль, небезопасное криптографирование, утечка персональных данных). После определения функции, сканер запускается для файла `auth_module.py`.

In [ ]:
import os
import re

# 1. СОЗДАЕМ ПОЛНЫЙ ТЕСТОВЫЙ ФАЙЛ С УЯЗВИМОСТЯМИ
full_test_code = """
import hashlib
import sqlite3
import os

def login_user(username, password):
    # Тест 1: Жестко прописанный пароль (Hardcoded)
    admin_password = "SuperSecretAdminPassword2026!"

    # Тест 2: Уязвимый MD5
    hashed_password = hashlib.md5(password.encode()).hexdigest()

    if username == "admin" and password == admin_password:
        return "Админ вошел"
    return f"User {username}"

def get_user_profile(user_input_id):
    # Тест 4: SQL-инъекция (прямая конкатенация строк)
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    query = f"SELECT * FROM profiles WHERE id = '{user_input_id}'"
    cursor.execute(query)
    return cursor.fetchone()

def execute_system_command(user_command):
    # Тест 5: Опасная функция os.system (риск Command Injection)
    os.system(user_command)

def SaveUserData(user_id, phone, inn):
    # Тест 3: Утечка ИИН и телефона в логи
    print(f"[LOG] ИИН={inn}, Телефон={phone}")

# Тест 6: Криптографическая функция без обработки исключений (try-except)
def decrypt_sensitive_data(crypto_payload):
    decrypted_data = crypto_payload.decode('utf-8')
    return decrypted_data
"""

with open("final_test_module.py", "w", encoding="utf-8") as f:
    f.write(full_test_code)

# 2. ПОЛНЫЙ КОРРЕКТНЫЙ СКАНЕР БЕЗОПАСНОСТИ (ВСЕ 6 ТЕСТОВ)
def run_comprehensive_security_test(file_path):
    print("=" * 85)
    print(f"ЗАПУСК КОМПЛЕКСНОГО ИСПЫТАНИЯ БЕЗОПАСНОСТИ ДЛЯ ФАЙЛА: {file_path}")
    print("=" * 85)

    if not os.path.exists(file_path):
        print("Файл не найден.")
        return

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
        content = "".join(lines)

    issues_found = 0

    # Регулярные выражения для построчных тестов (1-5)
    rules = {
        "1. Hardcoded Password": re.compile(r"(password|passwd|secret)\s*=\s*['\"][^'\"]+['\"]", re.IGNORECASE),
        "2. Insecure Crypto (MD5/SHA1)": re.compile(r"hashlib\.(md5|sha1)\("),
        "3. Personal Data Leak in Logs": re.compile(r"print\(.*(иин|inn|phone|паспорт|f\s*['\"].*\{.*\}).*"),
        "4. SQL Injection Risk (Concatenation)": re.compile(r"execute\(\s*f['\"].*\{\s*\w+\s*\}.*['\"]\s*\)"),
        "5. Dangerous System Call (RCE Risk)": re.compile(r"(os\.system|eval|exec)\("),
    }

    # Анализируем строки для тестов 1-5
    for line_num, line in enumerate(lines, 1):
        line_clean = line.strip()
        if line_clean.startswith("#") or line_clean.startswith("def "):
            continue

        for rule_name, pattern in rules.items():
            if pattern.search(line_clean):
                print(f"[КРИТИЧЕСКАЯ УЯЗВИМОСТЬ] -> {rule_name}")
                print(f"Строка {line_num}: {line_clean}")
                print("-" * 85)
                issues_found += 1

    # Тест 6: Поиск функций (содержащих decrypt, crypto, cipher, auth) без try-except
    crypto_functions = re.findall(r"def\s+(\w*(?:decrypt|crypto|cipher|auth)\w*)\s*\(", content)
    for func_name in crypto_functions:
        # Извлекаем тело функции до следующего def или конца файла
        func_match = re.search(r"def\s+" + re.escape(func_name) + r"[\s\S]*?(?=(?:def\s+|$))", content)
        if func_match:
            func_body = func_match.group(0)
            if "try:" not in func_body:
                print(f"[ЗАМЕЧАНИЕ ПО БЕЗОПАСНОСТИ] -> 6. Missing Exception Handling in Crypto Function")
                print(f"В функции '{func_name}' отсутствует блок try-except для защиты от утечки стека ошибок.")
                print("-" * 85)
                issues_found += 1

    print(f"ИТОГ РАСШИРЕННЫХ ИСПЫТАНИЙ: Обнаружено дефектов: {issues_found}")
    if issues_found > 0:
        print("СТАТУС: Проверка НЕ ПРОЙДЕНА. Код содержит критические уязвимости.")
    else:
        print("СТАТУС: Проверка успешно пройдена.")
    print("=" * 85)

# Запуск по полному файлу
run_comprehensive_security_test("final_test_module.py")

ЗАПУСК КОМПЛЕКСНОГО ИСПЫТАНИЯ БЕЗОПАСНОСТИ ДЛЯ ФАЙЛА: final_test_module.py
[КРИТИЧЕСКАЯ УЯЗВИМОСТЬ] -> 1. Hardcoded Password
Строка 8: admin_password = "SuperSecretAdminPassword2026!"
-------------------------------------------------------------------------------------
[КРИТИЧЕСКАЯ УЯЗВИМОСТЬ] -> 2. Insecure Crypto (MD5/SHA1)
Строка 11: hashed_password = hashlib.md5(password.encode()).hexdigest()
-------------------------------------------------------------------------------------
[КРИТИЧЕСКАЯ УЯЗВИМОСТЬ] -> 5. Dangerous System Call (RCE Risk)
Строка 27: os.system(user_command)
-------------------------------------------------------------------------------------
[КРИТИЧЕСКАЯ УЯЗВИМОСТЬ] -> 3. Personal Data Leak in Logs
Строка 31: print(f"[LOG] ИИН={inn}, Телефон={phone}")
-------------------------------------------------------------------------------------
[ЗАМЕЧАНИЕ ПО БЕЗОПАСНОСТИ] -> 6. Missing Exception Handling in Crypto Function
В функции 'decrypt_sensitive_data' отсутствует блок

### Результат комплексного SAST-сканирования `final_test_module.py`

Комплексный сканер обнаружил 5 дефектов в `final_test_module.py`, включая жестко прописанный пароль, небезопасное криптографирование, опасные системные вызовы, утечку персональных данных и отсутствие обработки исключений в криптографической функции. Это демонстрирует способность сканера выявлять различные типы уязвимостей в коде.

### III. Создание и запуск комплексного SAST-сканера на расширенном тестовом модуле

Этот блок кода сначала создает новый, более сложный тестовый файл `final_test_module.py`, включающий дополнительные типы уязвимостей, такие как SQL-инъекции, выполнение опасных системных команд и отсутствие обработки исключений в криптографических функциях. Затем определяется и запускается функция `run_comprehensive_security_test`, которая выполняет более полный анализ безопасности, проверяя на все 6 категорий уязвимостей.